# Entraînement du recommender CPF

Notebook reproductible pour construire un petit échantillon, inspecter les paires et lancer une démonstration légère.

In [1]:
from pathlib import Path
import pandas as pd
from deepforma.training.cpf_dataset import load_jsonl, split_by_group, summarize_rows
from deepforma.recommendation.training_recommender import TrainingRecommender, RecommenderConfig


In [3]:
formations = pd.read_parquet(Path('data/processed/cpf/formations_with_skills.parquet')).head(50)
pairs = load_jsonl(Path('data/training/cpf_pairs.jsonl'))[:20] if Path('data/training/cpf_pairs.jsonl').exists() else []
print('formations', len(formations))
print('pairs', len(pairs))
if pairs:
    print(pairs[0])


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/cpf/formations_with_skills.parquet'

In [ ]:
if pairs:
    summary = summarize_rows(pairs)
    print(summary)
    splits = split_by_group(pairs, seed=42)
    print({k: len(v) for k, v in splits.items()})


In [ ]:
if not formations.empty:
    metadata = formations.rename(columns={'skills_normalized': 'skills_normalized'})
    recommender = TrainingRecommender(metadata.to_dict(orient='records'), config=RecommenderConfig(limit=3))
    results = recommender.recommend({
        'target_job': formations.iloc[0]['title'],
        'user_skills': [],
        'missing_skills': [s.get('canonical_label') if isinstance(s, dict) else s for s in (formations.iloc[0]['skills_normalized'] or [])][:3],
        'desired_skills': [],
        'region_code': formations.iloc[0].get('region_code'),
        'department_code': formations.iloc[0].get('department_code'),
        'remote_allowed': True,
        'limit': 3,
    })
    results[:1]
